In [53]:
import sys
import os
sys.path.append(os.path.abspath('/acne-lds/model'))
sys.path.append(os.path.abspath('model'))
sys.path.append(os.path.abspath("/acne-lds/utils"))

In [54]:
from model_ld_smoothing import AcneModel

In [70]:
from predict_on_img import ModelInit
from PIL import Image

model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')
# img = Image.open(PATH_TO_IMAGE)
# predictions = model.predict_on_img(img)

In [56]:
# Assume ModelInit is imported and your checkpoint path is correct
import torch
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model = model_wrapper.model
model.eval()

# Dummy input tensor (use actual input size expected, e.g. 3x224x224)
example_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)

In [57]:
import torch
from torch import nn
from collections import namedtuple

from torchvision import transforms
# Define a namedtuple for outputs
ModelOutput = namedtuple("ModelOutput", ["cls_pred", "lesion_count"])

class ModelWrapper(nn.Module):
    def __init__(self, model, model_type="model_ld_smoothing"):
        super().__init__()
        self.model = model
        self.model_type = model_type

    def forward(self, x):
        cls, cou, cou2cls = self.model(x)

        if self.model_type == "model_ld_smoothing":
            cls = torch.stack(
                (
                    torch.sum(cls[:, :1], 1),
                    torch.sum(cls[:, 1:4], 1),
                    torch.sum(cls[:, 4:10], 1),
                    torch.sum(cls[:, 10:], 1),
                ),
                dim=1,
            )

        cls_pred = torch.argmax(0.5 * (cls + cou2cls), dim=1)
        lesion_count = torch.argmax(cou, dim=1) + 1

        return ModelOutput(cls_pred, lesion_count)

# Wrap your model
# Initialize model (make sure the path is correct)
model_wrapper = ModelInit(model_type="model_ld_smoothing", path_checkpoint="model/lds-weights/model_fold_4.pth", device="cpu")
model_2 = model_wrapper.model
model_2.eval()

wrapped_model = ModelWrapper(model_2)
wrapped_model.eval()
# Then trace the wrapped model

traced_model = torch.jit.trace(wrapped_model, example_input)

In [58]:
import coremltools as ct
image_input = ct.ImageType(
    name="input_image",
    shape=(1, 3, 224, 224),                 
    scale=1.0/255.0/0.226,   # list of 3 scales (1/std per channel)
    bias=[-0.45815152/0.2814769, 
          -0.361242  /0.226306, 
          -0.29348266/0.20132513],                  
    color_layout=ct.colorlayout.RGB,)

coreml_model = ct.convert(
    traced_model,
    convert_to="neuralnetwork",
    inputs=[ image_input ],
)
# coreml_model.save("AcneClassification.mlmodel")

Tuple detected at graph output. This will be flattened in the converted model.
Running MIL default pipeline:   0%|          | 0/69 [00:00<?, ? passes/s]/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:267: UserWarning: Output, '910', of the source model, has been renamed to 'var_910' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/opt/anaconda3/envs/pl-env/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:267: UserWarning: Output, '916', of the source model, has been renamed to 'var_916' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_neuralnetwork pipeline:   0%|          | 0/9 [00:00<?, ? passes/s]Output var var_910 of type int32 in function main is cast to type fp32
Output var var_916 of type int32 in function main is cast to type fp32
Translating MIL ==> NeuralNetwork Ops: 100%|██████████| 527/527 [00:17<00:00, 30.09 ops/

In [59]:
coreml_model_lut = ct.models.neural_network.quantization_utils.quantize_weights(
    coreml_model,
    nbits=8, 
    quantization_mode="linear_lut"
)

Quantizing using linear_lut quantization
Optimizing Neural Network before Quantization:
Finished optimizing network. Quantizing neural network..
Quantizing layer input.3 of type convolution
Quantizing layer input.11 of type convolution
Quantizing layer input.17 of type convolution
Quantizing layer out.1 of type convolution
Quantizing layer residual.1 of type convolution
Quantizing layer input.31 of type convolution
Quantizing layer input.37 of type convolution
Quantizing layer out.3 of type convolution
Quantizing layer input.49 of type convolution
Quantizing layer input.55 of type convolution
Quantizing layer out.5 of type convolution
Quantizing layer input.67 of type convolution
Quantizing layer input.73 of type convolution
Quantizing layer out.7 of type convolution
Quantizing layer residual.3 of type convolution
Quantizing layer input.87 of type convolution
Quantizing layer input.93 of type convolution
Quantizing layer out.9 of type convolution
Quantizing layer input.105 of type conv

In [60]:
coreml_model_lut.save("AcneClassQuantFin.mlpackage")

In [64]:
img = Image.open('../data/acne04-1/train/images/levle1_143_jpg.rf.509bf1d45ba1d1f6848d1f076bc44863.jpg')

In [65]:
from PIL import Image
import numpy as np
img_resized = img.resize((224, 224))

In [66]:
outputs = coreml_model.predict({"input_image": img_resized})

In [67]:
outputs.values()

dict_values([array([9.], dtype=float32), array([1.], dtype=float32)])

In [68]:
coreml_model_lut.predict({"input_image": img_resized})

{'var_916': array([9.], dtype=float32), 'var_910': array([1.], dtype=float32)}

In [71]:
cls, cou, cou2cls = model.predict_on_img(img)

## model comparisons

In [78]:
!pip install ultralytics

  Using cached ultralytics-8.3.124-py3-none-any.whl.metadata (37 kB)
  Using cached opencv_python-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata (20 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.4 MB/s eta 0:00:00
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.7 MB/s eta 0:00:00a 0:00:01
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 2.4 MB/s eta 0:00:00a 0:00:01
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached ultralytics-8.3.124-py3

In [79]:
from ultralytics import YOLO
yolo_model = YOLO("../notebooks/runs/detect/yolov11m_acne04/weights/best.pt")

In [83]:
quantised_model = ct.models.MLModel("../app/SkinSnap/SkinSnap/AcneClassQuantFin.mlpackage")
## grouth truth
gt_model = ModelInit(path_checkpoint='model/lds-weights/model_fold_4.pth')

In [85]:
# size, inference speed and accuracy

for images in ["test", "valid"]:
    for image in os.listdir(f"../data/acne04-1/{images}/images"):
        img = Image.open(f"../data/acne04-1/{images}/images/{image}")
        
        # img_resized = img.resize((224, 224))
        # outputs = quantised_model.predict({"input_image": img_resized})
        # cls, cou, cou2cls = gt_model.predict_on_img(img)
        # print(outputs.values())
        # print(cls, cou, cou2cls)